In [1]:
# Blibliotecas
import pandas as pd 
import numpy as np 
import os 
import random

from datetime import datetime, timedelta
from faker import Faker 

In [2]:
# Configuração da semente para reprodução do portfólio
Faker.seed(42)
np.random.seed(42)
random.seed(42)


In [3]:
class UserGenerator:
    """
    Classe responsãvel por gerar os dados da dimensão de usuários
    """
    def __init__(self, num_users: int):
        self.num_users = num_users
        self.fake = Faker(['pt_BR'])

    def generate_users(self) -> pd.DataFrame:
        users = []
        preferencias_jogo = ['esportes', 'cassino', 'misto']

        for _ in range(self.num_users):
            user_id = self.fake.unique.random_int(min=100000, max=999999)
            data_cadastro = self.fake.date_between(start_date='-6m', end_date='-3m')

            users.append({"user_id": user_id,
                          "nome": self.fake.name(),
                          "idade": random.randint(18, 65),
                          "estado": self.fake.state_abbr(),
                          "data_cadastro": data_cadastro,
                          "preferencia_jogo": random.choices(preferencias_jogo, weights=[0.5, 0.3, 0.2])[0],
                          "flag_vip": random.choices([0, 1], weights=[0.95, 0.05])[0]})
            
        return pd.DataFrame(users)

In [4]:
class TransactionGenerator:
    """
    Classe responsável por gerar as transações dos usuários 
    """

    def __init__(self, df_users: pd.DataFrame):
        self.df_users = df_users

    def generate_transactions(self) -> pd.DataFrame:
        transactions = []
        tipos_transacao = ['deposito', 'saque', 'aposta_esportiva', 'aposta_cassino']
        end_date = datetime.now()

        for _, user in self.df_users.iterrows():
            user_id = user['user_id']
            is_vip = user['flag_vip']

            # Distribuição de volume de apostas conforme o perfil do cliente
            num_transacoes = random.randint(50, 300) if is_vip else random.randint(5, 80)

            # Define propensão ao vício/risco para posterior análise de Jogo Responsável
            comportamento_risco = random.choices([True, False], weights=[0.15, 0.85])[0]

            # Inicia o relógio a partir da data de cadastro do usuário
            current_time = datetime.combine(user['data_cadastro'], datetime.min.time())

            for _ in range(num_transacoes):
                if current_time > end_date:
                    break

                tipo = random.choices(tipos_transacao, weights=[0.2, 0.1, 0.4, 0.3])[0]
                valor = float(np.random.exponential(scale=150.0)) if is_vip else float(np.random.exponential(scale=30.0))
                valor = round(max(5.0, valor), 2)

                valor_ganho = 0.0
                valor_perdido = 0.0

                # Regra de negócio: Simulação da margem matemática da casa
                if tipo in ['aposta_esportiva', 'aposta_cassino']:
                    resultado_ganhou = random.choices([True, False], weights=[0.42, 0.58])[0]
                    if resultado_ganhou:
                        valor_ganho = round(valor * random.uniform(1.2, 3.5), 2)
                    else:
                        valor_perdido = valor

                transactions.append({
                    "transacao_id": Faker().unique.uuid4(),
                    "user_id": user_id,
                    "data_hora": current_time,
                    "tipo_transacao": tipo,
                    "valor_transacao": valor,
                    "valor_ganho": valor_ganho,
                    "valor_perdido": valor_perdido
                })
                
                # Relógio avança incrementalmente de forma limpa apenas ao fim do ciclo
                horas_avanco = random.randint(1, 48) if not comportamento_risco else random.randint(1, 6)
                current_time += timedelta(hours=horas_avanco)
                
        return pd.DataFrame(transactions)

In [5]:
class DataExporter:
    """
    Classe responsável por gerenciar a persistência dos dados 
    """
    @staticmethod
    def export_to_csv(df: pd.DataFrame, directory: str, filename: str):
        if not os.path.exists(directory):
            os.makedirs(directory)
        path = os.path.join(directory, filename)
        df.to_csv(path, index=False)
        print(f"Arquivo {filename} exportado com sucesso em: {path}")

if __name__ == "__main__":
    print("Iniciando a geração de dados sintéticos...")

    # 1. Instancia e gera a dimensão de usuários
    user_factory = UserGenerator(num_users=10000)
    df_dim_usuarios = user_factory.generate_users()

    # 2. Instancia e gera a fato de transações
    transaction_factory = TransactionGenerator(df_dim_usuarios)
    df_fato_transacoes = transaction_factory.generate_transactions()

    # 3. Exportação estruturada para o Data Lake local (Camada Raw)
    raw_dir = os.path.join("data", "raw")
    DataExporter.export_to_csv(df_dim_usuarios, raw_dir, "dim_usuarios.csv")
    DataExporter.export_to_csv(df_fato_transacoes, raw_dir, "fato_transacoes.csv")

    print("Geração de dados finalizado com sucesso!")

Iniciando a geração de dados sintéticos...
Arquivo dim_usuarios.csv exportado com sucesso em: data\raw\dim_usuarios.csv
Arquivo fato_transacoes.csv exportado com sucesso em: data\raw\fato_transacoes.csv
Geração de dados finalizado com sucesso!
